In [ ]:
# import os
# import shutil
# import zipfile
# import kagglehub

# # 1️⃣ Current working directory
# cwd = os.getcwd()
# print("Current folder:", cwd)

# # 2️⃣ Download dataset
# path = kagglehub.dataset_download("imtkaggleteam/book-recommendation-good-book-api")
# print("Downloaded dataset path:", path)

# # 3️⃣ Extract to current folder
# if path.endswith(".zip"):
#     with zipfile.ZipFile(path, 'r') as zip_ref:
#         zip_ref.extractall(cwd)
#     print("Dataset extracted to:", cwd)
#     # 4️⃣ Delete original zip
#     os.remove(path)
#     print("Original zip deleted.")
# else:
#     # If it's a folder, copy to current folder
#     dest_path = os.path.join(cwd, "book-recommendation-good-book-api")
#     shutil.copytree(path, dest_path)
#     print("Dataset copied to:", dest_path)
#     # 4️⃣ Delete original folder
#     shutil.rmtree(path)
#     print("Original folder deleted.")

Current folder: d:\aiLearning\tuteDude_learning\assignment20


100%|██████████| 622k/622k [00:03<00:00, 189kB/s]

Extracting files...
Downloaded dataset path: C:\Users\aryam\.cache\kagglehub\datasets\imtkaggleteam\book-recommendation-good-book-api\versions\2
Dataset copied to: d:\aiLearning\tuteDude_learning\assignment20\book-recommendation-good-book-api
Original folder deleted.


In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("books.csv", engine='python', on_bad_lines='skip')
print(df.head())
print(df.info())
df=df[df['language_code']=='eng']
print("Shape after filtering for English books:", df.shape)
print(df.head())
df = df[["bookID","title", "authors"]]

print(df.head())
df.isna().sum()

   bookID                                              title  \
0       1  Harry Potter and the Half-Blood Prince (Harry ...   
1       2  Harry Potter and the Order of the Phoenix (Har...   
2       4  Harry Potter and the Chamber of Secrets (Harry...   
3       5  Harry Potter and the Prisoner of Azkaban (Harr...   
4       8  Harry Potter Boxed Set  Books 1-5 (Harry Potte...   

                      authors  average_rating        isbn         isbn13  \
0  J.K. Rowling/Mary GrandPré            4.57  0439785960  9780439785969   
1  J.K. Rowling/Mary GrandPré            4.49  0439358078  9780439358071   
2                J.K. Rowling            4.42  0439554896  9780439554893   
3  J.K. Rowling/Mary GrandPré            4.56  043965548X  9780439655484   
4  J.K. Rowling/Mary GrandPré            4.78  0439682584  9780439682589   

  language_code    num_pages  ratings_count  text_reviews_count  \
0           eng          652        2095690               27591   
1           eng         

bookID     0
title      0
authors    0
dtype: int64

In [18]:
print("Shape of dataset:", df.shape)

Shape of dataset: (0, 3)


In [27]:
df['features'] = df['title'] + " " + df['authors']
df.sample(5)


,bookID,title,authors,features
4662,16776,I Dreamed I Married Perry Mason (A Cece Caruso...,Susan Kandel,I Dreamed I Married Perry Mason (A Cece Caruso...
7957,30517,Colonialism and Neocolonialism,Jean-Paul Sartre/Azzedine Haddour,Colonialism and Neocolonialism Jean-Paul Sartr...
1355,4704,Ocean Star Express,Mark Haddon/Peter Sutton,Ocean Star Express Mark Haddon/Peter Sutton
2122,7664,A New Collection of Three Complete Novels: Con...,Michael Crichton,A New Collection of Three Complete Novels: Con...
7741,29907,Republic,Plato/Benjamin Jowett/Elizabeth Watson Scharff...,Republic Plato/Benjamin Jowett/Elizabeth Watso...


In [28]:
import pandas as pd
import numpy as np
import re
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nlp_preprocess(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercasing + Noise removal (from Task 3)
    text = text.lower()
    text = re.sub(r'http[s]?://\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = ' '.join(text.split())
    
    # 2. Stopword removal
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    
    # 3. Lemmatization (preferred over stemming)
    lemmatized = [lemmatizer.lemmatize(word) for word in words]
    
    return ' '.join(lemmatized)

# Apply final pipeline
df['final_clean_text'] = df['features'].apply(nlp_preprocess)
df.sample(5)

,bookID,title,authors,features,final_clean_text
1423,4956,How We Are Hungry: Stories,Dave Eggers,How We Are Hungry: Stories Dave Eggers,hungry story dave egger
4796,17247,Macbeth (No Fear Shakespeare),William Shakespeare/SparkNotes/John Crowther,Macbeth (No Fear Shakespeare) William Shakespe...,macbeth fear shakespeare william shakespearesp...
3449,12577,The Collected Stories,Eudora Welty,The Collected Stories Eudora Welty,collected story eudora welty
4801,17255,McCoy: The Provenance Of Shadows (Star Trek: C...,David R. George III,McCoy: The Provenance Of Shadows (Star Trek: C...,mccoy provenance shadow star trek crucible 1 d...
10451,42547,The Autobiography of Martin Luther King Jr.,Martin Luther King Jr./Clayborne Carson,The Autobiography of Martin Luther King Jr. M...,autobiography martin luther king jr martin lut...


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,      # Reasonable limit to avoid too many dimensions
    ngram_range=(1, 2),     # Unigrams + bigrams for better context
    stop_words='english',
    min_df=2                # Ignore very rare terms
)

# Fit and transform the cleaned text
tfidf_matrix = tfidf.fit_transform(df['final_clean_text'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Number of features (vocabulary size):", len(tfidf.get_feature_names_out()))

TF-IDF Matrix Shape: (8906, 5000)
Number of features (vocabulary size): 5000


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

# Brief explanation:
print("\nWhy Cosine Similarity?")
print("- It measures the cosine of the angle between two vectors.")
print("- It focuses on the **direction** (orientation) of the vectors rather than their magnitude.")
print("- This is ideal for text data because documents can have different lengths, but we care about the **relative importance** of terms (TF-IDF already normalizes for this).")
print("- Values range from 0 (completely different) to 1 (identical content).")

Cosine Similarity Matrix Shape: (8906, 8906)

Why Cosine Similarity?
- It measures the cosine of the angle between two vectors.
- It focuses on the **direction** (orientation) of the vectors rather than their magnitude.
- This is ideal for text data because documents can have different lengths, but we care about the **relative importance** of terms (TF-IDF already normalizes for this).
- Values range from 0 (completely different) to 1 (identical content).


In [ ]:
def recommend(item_name, top_n=5):
    """
    Recommend top N similar books based on the 'title' column.
    """
    # Step 1: Find the index of the selected item
    try:
        idx = df[df['title'].str.contains(item_name, case=False, na=False)].index[0]
    except IndexError:
        return f"Book '{item_name}' not found in the dataset."
    
    # Step 2: Get similarity scores for this item with all others
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Step 3: Sort the books based on similarity scores (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Exclude the item itself and take top_n
    sim_scores = sim_scores[1:top_n+1]
    
    # Get the book indices
    book_indices = [i[0] for i in sim_scores]
    
    # Return the most similar books with scores
    recommendations = df.iloc[book_indices][['title', 'authors']].copy()
    recommendations['similarity_score'] = [score[1] for score in sim_scores]
    
    return recommendations.reset_index(drop=True)


# ========================
# Testing the function with 3 different items
# ========================

test_items = [
    "How We Are Hungry",
    "Macbeth",
    "Harry Potter and the Half-Blood Prince"
]

print("=== BOOK RECOMMENDATIONS ===\n")

for item in test_items:
    print(f"Recommendations for: **{item}**")
    print("-" * 50)
    recs = recommend(item, top_n=5)
    print(recs)
    print("\n")

=== BOOK RECOMMENDATIONS ===

Recommendations for: **How We Are Hungry**
--------------------------------------------------
                                               title  \
0             The Wreath (Kristin Lavransdatter  #1)   
1              The Cross (Kristin Lavransdatter  #3)   
2  Kristin Lavransdatter (Kristin Lavransdatter  ...   
3                                              Jenny   
4                    Mitz The Marmoset of Bloomsbury   

                                          authors  similarity_score  
0                    Sigrid Undset/Tiina Nunnally          0.939519  
1  Sigrid Undset/Tiina Nunnally/Sherrill Harbison          0.886677  
2    Sigrid Undset/Tiina Nunnally/Brad Leithauser          0.876933  
3                    Sigrid Undset/Tiina Nunnally          0.555770  
4                                    Sigrid Nunez          0.349132  


Recommendations for: **Macbeth**
--------------------------------------------------
                                 

: 